In [3]:
import os

# Maak powerbi folder aan als die niet bestaat
os.makedirs('../data/powerbi', exist_ok=True)

print("✅ PowerBI folder created/verified!")

✅ PowerBI folder created/verified!


In [4]:
import pandas as pd
import numpy as np

print("="*60)
print("PREPARING DATA FOR POWER BI")
print("="*60)

# Laad de complete dataset
df = pd.read_csv('../data/processed/transactions_with_anomalies.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Total transactions: {len(df)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

# Selecteer en hernoem kolommen voor Power BI (duidelijke namen)
powerbi_df = df[[
    'Date',
    'Amount',
    'Balance',
    'Counterparty',
    'Final_Category',
    'Is_Income',
    'Is_Expense',
    'Amount_Abs',
    'Year',
    'Month',
    'Month_Name',
    'Day',
    'Day_Name',
    'Is_Weekend',
    'Quarter',
    'Is_Anomaly_Either',
    'Prediction_Confidence'
]].copy()

# Hernoem kolommen naar duidelijke namen
powerbi_df.columns = [
    'Date',
    'Amount',
    'Balance',
    'Merchant',
    'Category',
    'Is_Income',
    'Is_Expense',
    'Amount_Absolute',
    'Year',
    'Month_Number',
    'Month_Name',
    'Day',
    'Day_Name',
    'Is_Weekend',
    'Quarter',
    'Is_Anomaly',
    'ML_Confidence'
]

# Maak extra nuttige kolommen
powerbi_df['Year_Month'] = powerbi_df['Date'].dt.to_period('M').astype(str)
powerbi_df['Week_Number'] = powerbi_df['Date'].dt.isocalendar().week

# Income/Expense label (voor filtering)
powerbi_df['Transaction_Type'] = powerbi_df.apply(
    lambda x: 'Income' if x['Is_Income'] == 1 else 'Expense', axis=1
)

# Anomaly label
powerbi_df['Anomaly_Status'] = powerbi_df['Is_Anomaly'].apply(
    lambda x: 'Anomaly' if x == 1 else 'Normal'
)

print("\n✅ Data prepared for Power BI!")
print(f"\nFinal shape: {powerbi_df.shape}")
print(f"Columns: {powerbi_df.columns.tolist()}")

# Save voor Power BI
powerbi_df.to_csv('../data/powerbi/transactions_powerbi.csv', index=False)
print(f"\n✅ Saved: data/powerbi/transactions_powerbi.csv")

print("\nSample data:")
print(powerbi_df.head())

PREPARING DATA FOR POWER BI
Total transactions: 1505
Date range: 2024-12-08 to 2025-12-08

✅ Data prepared for Power BI!

Final shape: (1505, 21)
Columns: ['Date', 'Amount', 'Balance', 'Merchant', 'Category', 'Is_Income', 'Is_Expense', 'Amount_Absolute', 'Year', 'Month_Number', 'Month_Name', 'Day', 'Day_Name', 'Is_Weekend', 'Quarter', 'Is_Anomaly', 'ML_Confidence', 'Year_Month', 'Week_Number', 'Transaction_Type', 'Anomaly_Status']

✅ Saved: data/powerbi/transactions_powerbi.csv

Sample data:
        Date  Amount  Balance                           Merchant     Category  \
0 2024-12-08  -25.00   180.36             CCV*Cuijkse Brouwbriga  other_shops   
1 2024-12-08  -60.00    20.36  R. Pittens via Rabo Betaalverzoek    transport   
2 2024-12-09   -1.74    18.62                  Albert Heijn 1382  supermarket   
3 2024-12-10   -3.76    14.86                  Albert Heijn 1382  supermarket   
4 2024-12-11  390.09   404.95                   M&L Italian B.V.       salary   

   Is_Income  Is

In [5]:
print("="*60)
print("PREPARING PREDICTIONS FOR POWER BI")
print("="*60)

# Laad predictions
predictions = pd.read_csv('../data/processed/spending_predictions.csv')

# Hernoem voor duidelijkheid
predictions_powerbi = predictions[[
    'Category',
    'Predicted_Next_Month',
    'Historical_Average',
    'Last_Month_Actual'
]].copy()

predictions_powerbi.columns = [
    'Category',
    'Predicted_Amount',
    'Historical_Average',
    'Last_Month'
]

# Bereken % verschil
predictions_powerbi['Change_vs_Average'] = (
    (predictions_powerbi['Predicted_Amount'] / predictions_powerbi['Historical_Average'] - 1) * 100
)

print(f"✅ Predictions prepared!")
print(f"Categories: {len(predictions_powerbi)}")

# Save
predictions_powerbi.to_csv('../data/powerbi/predictions_powerbi.csv', index=False)
print(f"✅ Saved: data/powerbi/predictions_powerbi.csv")

print("\nSample predictions:")
print(predictions_powerbi.head())

PREPARING PREDICTIONS FOR POWER BI
✅ Predictions prepared!
Categories: 10
✅ Saved: data/powerbi/predictions_powerbi.csv

Sample predictions:
      Category  Predicted_Amount  Historical_Average  Last_Month  \
0         rent        423.696970          411.333333      417.00   
1     bar_cafe        426.036212          355.856667      591.09   
2    transport        330.873846          272.032308       64.88   
3   restaurant        228.254231          212.978462       22.85   
4  supermarket        190.646154          199.619231       96.50   

   Change_vs_Average  
0           3.005746  
1          19.721296  
2          21.630349  
3           7.172448  
4          -4.495096  


In [6]:
print("="*60)
print("CREATING SUMMARY STATISTICS FOR POWER BI")
print("="*60)

# Bereken key metrics
total_transactions = len(powerbi_df)
total_income = powerbi_df[powerbi_df['Is_Income'] == 1]['Amount'].sum()
total_expenses = powerbi_df[powerbi_df['Is_Expense'] == 1]['Amount_Absolute'].sum()
net_cashflow = total_income - total_expenses
current_balance = powerbi_df.sort_values('Date')['Balance'].iloc[-1]
avg_transaction = powerbi_df['Amount_Absolute'].mean()
total_anomalies = powerbi_df['Is_Anomaly'].sum()
unique_merchants = powerbi_df['Merchant'].nunique()

# Maak summary dataframe
summary_stats = pd.DataFrame({
    'Metric': [
        'Total Transactions',
        'Total Income',
        'Total Expenses',
        'Net Cashflow',
        'Current Balance',
        'Average Transaction',
        'Total Anomalies',
        'Unique Merchants'
    ],
    'Value': [
        total_transactions,
        total_income,
        total_expenses,
        net_cashflow,
        current_balance,
        avg_transaction,
        total_anomalies,
        unique_merchants
    ]
})

print(summary_stats)

# Save
summary_stats.to_csv('../data/powerbi/summary_stats.csv', index=False)
print(f"\n✅ Saved: data/powerbi/summary_stats.csv")

print("\n" + "="*60)
print("ALL POWER BI DATA PREPARED!")
print("="*60)
print("\nFiles created:")
print("  1. data/powerbi/transactions_powerbi.csv")
print("  2. data/powerbi/predictions_powerbi.csv")
print("  3. data/powerbi/summary_stats.csv")

CREATING SUMMARY STATISTICS FOR POWER BI
                Metric         Value
0   Total Transactions   1505.000000
1         Total Income  29750.970000
2       Total Expenses  29611.900000
3         Net Cashflow    139.070000
4      Current Balance     38.570000
5  Average Transaction     39.443767
6      Total Anomalies     76.000000
7     Unique Merchants    388.000000

✅ Saved: data/powerbi/summary_stats.csv

ALL POWER BI DATA PREPARED!

Files created:
  1. data/powerbi/transactions_powerbi.csv
  2. data/powerbi/predictions_powerbi.csv
  3. data/powerbi/summary_stats.csv
